# Feature Engineering

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [3]:
data = pd.read_csv('housing/housing.csv')
data.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [5]:
data['median_house_value'].describe()

count     20640.000000
mean     206855.816909
std      115395.615874
min       14999.000000
25%      119600.000000
50%      179700.000000
75%      264725.000000
max      500001.000000
Name: median_house_value, dtype: float64

## Feature Transformation

In [ ]:
# Scaling

from sklearn.preprocessing import StandardScaler 

scaler = StandardScaler()
df_scaled = scaler.fit_transform(data)
print(df_scaled)


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
data_minmax = scaler.fit_transform(data)
pritn(data_minmax)

In [ ]:
train_set, test_set = train_test_split(data, test_size=0.2, random_state=42)

X_train = train_set[['housing_median_age', 'median_income','median_house_value']]
X_test = test_set[['housing_median_age', 'median_income', 'median_house_value']]

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(X_train)

x_test_scaled = scaler.transform(X_test)

In [ ]:
# Encoding 

from sklearn.preprocessing import OneHotEncoder 
encoder = OneHotEncoder(sparse_output=False)
encoder = encoder.fit_transform(data['ocean_proximtiy'])
print(encoder)

In [ ]:
from sklearn.preprocessing import LabelEncoder 

encoder = LabelEncoder()
data['ocean_proximity'] = encoder.fit_transform(data['ocean_proximity'])
print(data)

In [ ]:
# Log Transform -> mengatasi data skewed

""" 
np.log -> mengubah angka besar menjadi lebih "kecil dan terkompres"
"""

data['housing_median_age'] = np.log(data['housing_median_age'])

In [ ]:
# Binning

data['kategori_harga_rumah'] = pd.cut(
    data['median_house_value'],
    bins= [14999, 119600, 179700, 264725, np.inf],
    labels= ['Kecil', 'Menengah', 'Atas', 'Tajir']
)

data['kategori_harga_rumah']

## Feature Creation

In [ ]:
""" 
Feature Creation -> membuat fitur baru dari fitur yang sudah ada
"""

In [ ]:
data['house_per_year'] = data['housing_median_age'] / data['median_house_value']

## Feature Selection

In [ ]:
""" 
| Metode   | Cara kerja  | Cepat | Akurat |
| -------- | ----------- | ----- | ------ |
| Filter   | statistik   | ✅    | ❌     |
| Wrapper  | coba model  | ❌    | ✅     |
| Embedded | dalam model | ✅    | ✅     |
"""

In [ ]:
df = pd.DataFrame({
    "luas": [50, 60, 70, 80, 90],
    "kamar": [2, 3, 3, 4, 5],
    "umur_rumah": [10, 15, 20, 25, 30],
    "dummy": [1, 1, 1, 1, 1],  # fitur tidak berguna
    "harga": [500, 600, 650, 800, 900]
})

X = df.drop("harga", axis=1)
y = df["harga"]

In [ ]:
# Filter Method (Correlation)

corr = df.corr()['harga'].drop('harga')

selected_features = corr[abs(corr) > 0.5].index 

x_selected = df['selected_features']

print('Fitur Terpilih:', selected_features.tolist())
print(x_selected)

In [ ]:
# Filter Method (SelectKBest)

from sklearn.feature_selection import SelectKBest, f_regression

selector = SelecKBest(score_func=f_regression, k=2)
x_new = selector.fit_transform(X, y)

print('Score tiap fitur:' selector.scores_)
print('Fiture terpilih:', X.columns[selector.get_support()])

In [ ]:
# Filter Method (Chi-square) -> Data kategori (klasifikasi) dan nilai tidak boleh negatif

from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import OneHotEncoder

df = pd.DataFrame({
    "gender": ["M", "F", "F", "M", "M"],
    "kota": ["A", "B", "A", "B", "A"],
    "beli": [1, 0, 1, 0, 1]
})


X = df[['gender', 'kota']]
y = df['beli']

# Encoding terlebih dahulu 
encoder = OneHotEncoder(sparse_output=False)
X_encoded = encoder.fit_transform(X)

# Chi-Square
selector = SelectKBest(score_func=chi2, k=2)
X_selected = selector.fit_transform(X_encoded, y)

print("Score:", selector.scores_)
print("Fitur terpilih:", selector.get_support())

In [ ]:
# Filter Method (ANOVA) -> Klasfikasi 

from sklearn.feature_selection import SelectKBest, f_classif 

df = pd.DataFrame({
    "gender": ["M", "F", "F", "M", "M"],
    "kota": ["A", "B", "A", "B", "A"],
    "beli": [1, 0, 1, 0, 1]
})

X = df[['nilai_math', 'nilai_bahasa']]
y = df['lulus']

selector = SelectKBest(score_func=f_classif, k=1)
X_selected = selector.fit_transform(X, y)

print("Score:", selector.scores_)
print('Fitur terpilih:', X.columns[selector.get_support()])



In [ ]:
# Filter Method (ANOVA) -> Regresi 

from sklearn.feature_selection import f_regression 

X = df[['luas', 'kamar']]
y = df['harga']

selector = SelectKBest(score_func=f_regression, k=1)
X_selected = selector.fit_transform(X, y)

print("Score:", selector.scores_)
print('Fitur terpilih', X.columns[selector.get_support()])

In [ ]:
# Wrapper Method (RFE)

from skleran.feature_selection import RFE
from sklearn.linear_model import LinearRegression 

model = LinearRegression()

selector = RFE(mode, n_features_to_select=2)
x_rfe = selector.fit_transform(X, y)

print('Fitur terpilih:', X.columns[selector.support_])

In [ ]:
# Wrapper Method (Sequential Feature Selection)

from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression 

model = LinearRegression()

sfs = SequentialFeatureSelector(model, n_features_to_select=2, direction="forward")
sfs.fit(X, y)

print('Fitur terpilih:', X.columns[sfs.get_support()])

In [ ]:
# Embedded Method (Lasso - L1 Regularization)

from sklearn.linear_model import Lasso 

lasso = Lasso(alpha=0.1)
lasso.fit(X, y)

print('Koefisien:', lasso.coef_)

selected_features = X.columns[lasso.coef_ != 0]

print("Fitur terpilih:", selected_features)

In [ ]:
# Embedded Method (Random Forest Importance)

from sklearn.ensemble import RandomForestRegressor 

model = RandomForestRegressor(random_state=42)
model.fit(X, y)

importances = model.feature_importances_


for name, score in zip(X.columns, importances):
    print(name, ":", score)


In [ ]:
# Embedded + Seleksi (SelectFromModel)

from sklearn.feature_selection import SelectFromModel 

selector = SelectFromModel(model, threshold='mean')
x_selected = selector.fit_transform(X, y)

print('Fitur Terpilih:', X.columns[selector.get_support()])

In [ ]:
# Variance Threshold

from sklearn.feature_selection import VarianceThreshold 

selector = VarianceThreshold(threshold=0.0)
X_var = selector.fit_transform(X)

print('Fitur terpilih:', X.columns[selector.get_support()])

## Feature Extraction

In [ ]:
# PCA (Principal Component Analysis)

import pandas as pd 
from sklearn.decomposition import PCA 
from sklearn.preprocessing import StandardScaler 

df = pd.DataFrame({
    "x1": [1, 2, 3, 4],
    "x2": [2, 4, 6, 8],  # sangat berkorelasi dengan x1
    "x3": [5, 6, 7, 8]
})

# scaling dulu
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# PCA (ambil 2 komponen)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

In [ ]:
# Kernel PCA (versi non-linear)

from sklearn.decomposition import KernelPCA 

kpca = KernelPCA(n_components=2, kernel='rbf')
X_kpca = kpca.fit_transform(X_scaled)
print(X_kpca)

In [ ]:
# LDA (Linear Discriminant Analysis) -> Klasifikasi

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA 

y = [0, 0, 1, 1]

lda = LDA(n_components=1)
X_lda = lda.fit_transform(X_scaled, y)

print(X_lda)

## Model as Feature Engineering

In [ ]:
# Prediksi sebagai Fitur Baru

import pandas as pd 
from sklearn.linear_model import LinearRegression 
from sklearn.ensemble import RandomForestRegressor 
from sklearn.model_selection import cross_val_predict

# Data
df = pd.DataFrame({
    "luas": [50, 60, 70, 80, 90],
    "kamar": [2, 3, 3, 4, 5],
    "harga": [500, 600, 650, 800, 900]
})

X = df[['luas', 'kamar']]
y = df['harga']

model_1 = LinearRegression()

df['pred_linear'] = cross_val_predict(model_1, X, y, cv=3)

X_new = df[['luas', 'kamar', 'pred_linear']]

model_2 = RandomForestRegressor(random_state=42)
model_2.fit(X_new, y)

print(df)